In [47]:
#!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [48]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()


print(f"Length of dataset in characters: {len(text):,}")


batch_size = 32
context_size = 8
max_iters = 10000
eval_interval = 300
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embedding_dim = 32

print (f"Using device: {device}")


Length of dataset in characters: 1,115,394
Using device: cuda


In [49]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [50]:
# map chars to ints

s_to_i = {ch: i for i, ch in enumerate(chars)}
i_to_s = {i: ch for i, ch in enumerate(chars)}


def encode(s: str) -> list[int]:
    ls = [s_to_i[c] for c in s]
    return ls


def decode(ls: list[int]) -> str:
    st = [i_to_s[i] for i in ls]
    st = "".join(st)
    return st


test_str = "She dont believe in shooting stars"

print(encode(test_str))
print(decode(encode(test_str)))

[31, 46, 43, 1, 42, 53, 52, 58, 1, 40, 43, 50, 47, 43, 60, 43, 1, 47, 52, 1, 57, 46, 53, 53, 58, 47, 52, 45, 1, 57, 58, 39, 56, 57]
She dont believe in shooting stars


In [51]:
data = torch.tensor(encode(text), dtype=torch.long)

print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [52]:
# train val
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [53]:
context_len = 8
train_data[: context_len + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [54]:
x = train_data[:context_len]
y = train_data[1 : context_len + 1]
for t in range(context_len):
    context = x[: t + 1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [55]:
def get_batch(split: str) -> tuple:
    data = train_data if split == "train" else val_data

    # random context size size data from data
    ix = torch.randint(len(data) - context_size, (batch_size,))

    # stack of 1d tensors for inputs and targets, they become a batch_size x context_size matrix (completely indpendent, just for eficiency )
    x = torch.stack([data[i : i + context_size] for i in ix])
    y = torch.stack([data[i + 1 : i + context_size + 1] for i in ix])
    
    x, y = x.to(device), y.to(device)
    
    return x, y


xb, yb = get_batch("train")
print("inputs: ")
print(xb.shape)
print(xb)

print("targets: ")
print(yb.shape)
print(yb)

print("---------")

for b in range(batch_size):  # batch dimension
    for t in range(context_size):  # time dimension
        context = xb[b, : t + 1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs: 
torch.Size([32, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54],
        [57, 43, 60, 43, 52,  1, 63, 43],
        [60, 43, 42,  8,  0, 25, 63,  1],
        [56, 42,  5, 57,  1, 57, 39, 49],
        [43, 57, 58, 63,  6,  1, 58, 46],
        [43,  1, 51, 39, 63,  1, 40, 43],
        [58, 46, 43,  1, 43, 39, 56, 57],
        [39, 58, 47, 53, 52, 12,  1, 37],
        [53, 56, 43,  1, 21,  1, 41, 39],
        [50, 39, 52, 63,  1, 47, 58, 57],
        [56, 53, 63,  1, 42, 47, 42,  1],
        [39, 51,  1, 39, 44, 56, 39, 47],
        [17, 24, 21, 38, 13, 14, 17, 32],
        [ 1, 39, 52, 42,  1, 45, 43, 50],
        [ 1, 58, 46, 39, 58,  1, 42, 53],
        [ 1, 61, 53, 59, 50, 42,  1, 21],
        [59, 57, 40, 39, 52, 42,  1, 40],
        [52, 42,  8,  0,  0, 23, 21, 26],
        [45, 53, 42, 57,  0, 23, 43, 43],
        [52,  1, 61, 39, 57,  1, 51, 53],
     

In [56]:
class Head(nn.Module):
    """ one head of self attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embedding_dim, head_size, bias=False)
        self.query = nn.Linear(n_embedding_dim, head_size, bias=False)
        self.value = nn.Linear(n_embedding_dim, head_size, bias=False)
        self.register_buffer("trill", torch.tril(torch.ones(context_size, context_size)))

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)  # (B,T,C)
        q = self.query(x)  # (B,T,C)
        
        # compute attention scores --- (B,T,C) @ (B,C,T) --> (B,T,T)
        wei = q @ k.transpose(-2, -1) * C**-0.5 
        wei= wei.masked_fill(self.trill[:T, :T] == 0, float("-inf")) 
        wei = F.softmax(wei, dim=-1) 
        
        v = self.value(x)  # (B,T,C)
        out = wei @ v  # (B,T,T) @ (B,T,C) --> (B,T,C)
        return out

In [57]:
class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embedding_dim, n_embedding_dim)
        
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [58]:
class FeedForward(nn.Module):
    
    def __init__(self, n_embedding_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embedding_dim, n_embedding_dim),
            nn.ReLU(),
            nn.Linear(n_embedding_dim, n_embedding_dim)
        )
    
    def forward(self, x):
        return self.net(x)

In [59]:
class TransformerBlock(nn.Module):
    
    def __init__(self, n_embedding_dim, num_heads):
        super().__init__()
        head_size = n_embedding_dim // num_heads
        self.sa_heads = MultiHeadAttention(num_heads, head_size) #4 heads of 8 dim self attention
        self.ffwd = FeedForward(n_embedding_dim)
        
    def forward(self, x):
        x = x + self.sa_heads(x)
        x = x + self.ffwd(x)
        return x

In [60]:

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embedding_dim)
        self.position_embedding_table = nn.Embedding(context_size, n_embedding_dim)
        self.transformer_blocks = nn.Sequential(
            TransformerBlock(n_embedding_dim, num_heads=4),
            TransformerBlock(n_embedding_dim, num_heads=4),
            TransformerBlock(n_embedding_dim, num_heads=4),
        )
        self.lm_head = nn.Linear(n_embedding_dim, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T) tensors of integers

        B,T = idx.shape
        
        tok_emb = self.token_embedding_table(idx) # (B,T,C) 
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        tok_emb = tok_emb + pos_emb # (B,T,C)
        
        tok_emb = self.transformer_blocks(tok_emb)  # transformer blocks
        
        logits = self.lm_head(tok_emb) 

        if targets == None:
            loss = None
        else:
            # reshape for pytorch
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indexes in the current context

        for _ in range(max_new_tokens):
            idx_cond = idx[:, -context_size:]  # crop to the last context_size tokens
            
            # get preds
            logits, loss = self(idx_cond)

            # only last step
            logits = logits[:, -1, :]  # (B, C)

            # probs
            probs = F.softmax(logits, dim=-1)  # (B, C)

            # sample from distribution (one prediction for each batch)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            
            idx = torch.cat((idx, idx_next), dim=1)  # (B, T+1)

        return idx


m = BigramLanguageModel()
m.to(device)

logits, loss = m(xb, yb)
print(logits.shape)
print(loss)


print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=100)[0].tolist()))


torch.Size([256, 65])
tensor(4.5444, device='cuda:0', grad_fn=<NllLossBackward0>)

WYC;xbRkRZk
dc.wf,ZTAOLXT-yCtK
b:iPWCmbBbUA$A:.YSGgO-33&M:c?KLTd
Py'YWdWtbNNNuyqBBC&G.tbfC dXl!DZaLe


In [61]:
#Train teh model 
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

for iter in range(max_iters):
    
    # get batch
    xb, yb = get_batch("train")

    # forward pass
    logits, loss = m(xb, yb)

    # backward pass
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())

1.9275952577590942


In [65]:
#out from trained model
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long, device=device), max_new_tokens=400)[0].tolist()))



Her, the to gain the boeld hold;
I the an nugght his if teny, delel!
Hose so, I he could new,
And erepens; fin justious happock, say, as;
folly thouk viresce!
Cout!.

PLOANNIO:
Bused well, sholk
Ere, whald.
Like hame, he knather there?
'er rother a rishn. Proth,
Gork a gay in his loft his, byodaster no, hithese have blay not,
Though call,
But with I heach ous chip, of mongles.

PLIUS:
But twith we
